In [1]:
import pandas as pd
import numpy as np
from src_RF_DT import *

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree
from sklearn.metrics import roc_auc_score
from sklearn.metrics import RocCurveDisplay
from matplotlib import pyplot as plt

# 1.0 - Classificação do Desempenho Baseado em Fatores Socioeconômicos Usando Random Forest

In [2]:
colunas = ['Q001','Q002','Q003', 'Q004', 'Q005', 'Q006', 'Q007', 'Q008', 'Q009', 'Q010', 'Q011', 'Q012', 'Q013', 'Q014', 'Q015', 'Q016', 'Q017', 'Q018', 
           'Q019', 'Q020', 'Q021', 'Q022', 'Q023', 'Q024', 'Q025', 'TP_PRESENCA_LC', 'TP_PRESENCA_CH', 'TP_PRESENCA_CN', 'TP_PRESENCA_MT', 
           'TP_FAIXA_ETARIA', 'TP_ESTADO_CIVIL', 'TP_ESCOLA', 'TP_ST_CONCLUSAO', 'IN_TREINEIRO', 
           'NU_ANO', 'TP_LOCALIZACAO_ESC','TP_SIT_FUNC_ESC', 'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO', 'TP_DEPENDENCIA_ADM_ESC']

df = pd.read_parquet("../data/enem_parquet", columns = colunas)

## 1.1 - Pré-Processamento dos Dados 

In [3]:
df = pre_processor_rf_dt(df, objetivo = 'Desempenho', n_samples = 50_000)

## 1.2 - Construção da Matriz X e Vetor y

In [4]:
X = df.drop(columns=['MEDIA', 'CLASSE', 'FALTOU'])

y = df['CLASSE']

## 1.3 - Separação em Dados de Treino e Teste

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

## 1.4 - Treinando o modelo

In [6]:
rf = RandomForestClassifier()

rf.fit(X_train, y_train)

print('Ein:  %0.4f' % (1 - accuracy_score(y_train, rf.predict(X_train))))
print('Eval: %0.4f' % (1 - accuracy_score(y_val,  rf.predict(X_val))))

print(classification_report(y_val, rf.predict(X_val)))

Ein:  0.0047
Eval: 0.3219
              precision    recall  f1-score   support

           0       0.67      0.71      0.69       881
           1       0.69      0.65      0.67       871

    accuracy                           0.68      1752
   macro avg       0.68      0.68      0.68      1752
weighted avg       0.68      0.68      0.68      1752



## 1.5 Treinando com os Melhores Parâmetros

In [7]:
cv_rf = tune_random_forest(X_train, y_train, n_iter=10, cv=5, scoring='accuracy', random_state=42)

print('Ein:  %0.4f' % (1 - accuracy_score(y_train, cv_rf.predict(X_train))))
print('Eout: %0.4f' % (1 - accuracy_score(y_val,  cv_rf.predict(X_val))))

print(classification_report(y_val, cv_rf.predict(X_val)))

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV] END max_depth=10, max_features=log2, min_samples_leaf=25, min_samples_split=10, n_estimators=60; total time=   0.1s
[CV] END max_depth=30, max_features=sqrt, min_samples_leaf=10, min_samples_split=50, n_estimators=60; total time=   0.1s
[CV] END max_depth=30, max_features=sqrt, min_samples_leaf=10, min_samples_split=50, n_estimators=60; total time=   0.2s
[CV] END max_depth=30, max_features=sqrt, min_samples_leaf=10, min_samples_split=50, n_estimators=60; total time=   0.1s
[CV] END max_depth=10, max_features=log2, min_samples_leaf=25, min_samples_split=10, n_estimators=60; total time=   0.2s
[CV] END max_depth=30, max_features=sqrt, min_samples_leaf=10, min_samples_split=50, n_estimators=60; total time=   0.2s
[CV] END max_depth=30, max_features=sqrt, min_samples_leaf=10, min_samples_split=50, n_estimators=60; total time=   0.2s
[CV] END max_depth=10, max_features=log2, min_samples_leaf=25, min_samples_split=10, n_estim

In [8]:
print(cv_rf.best_estimator_)

RandomForestClassifier(class_weight='balanced', max_depth=10,
                       max_features='log2', min_samples_leaf=25,
                       min_samples_split=10, n_estimators=60, random_state=42)


## 1.6 - Treinando com todos os dados de treino

In [9]:
rf = RandomForestClassifier(**cv_rf.best_params_, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf.fit(X_train, y_train)

print('Ein:  %0.4f' % (1 - accuracy_score(y_train, rf.predict(X_train))))
print('Eout: %0.4f' % (1 - accuracy_score(y_test,  rf.predict(X_test))))

print(classification_report(y_test, rf.predict(X_test)))


Ein:  0.2835
Eout: 0.2973
              precision    recall  f1-score   support

           0       0.68      0.77      0.72      1111
           1       0.73      0.63      0.68      1079

    accuracy                           0.70      2190
   macro avg       0.71      0.70      0.70      2190
weighted avg       0.71      0.70      0.70      2190

